In [1]:
import folium
import pandas as pd
import osmnx as ox
import networkx as nx
import numpy as np

In [2]:
routes = pd.read_csv('../data/solution_routes.csv')
sample = pd.read_csv('../data/sample_stops.csv')
distance_matrix = np.load('../data/distance_matrix.npy').astype(int)
G = ox.load_graphml('../data/raleigh_graph.graphml')
m = folium.Map(location = [35.8, -78.6], zoom_start=11)

In [3]:
print(sample.columns)

Index(['Address', 'City Limits', 'latitude', 'longitude', 'node_id', 'demand'], dtype='str')


In [4]:
colors = ['blue', 'red', 'green', 'purple', 'orange', 'darkred']
depot = routes[routes['stop_sequence'] == 0].iloc[0]
folium.Marker(
    location = [routes.iloc[0]['latitude'], routes.iloc[0]['longitude']],
    popup = 'DRT3 Depot - Amazon',
    icon = folium.Icon(color = 'black', icon = 'star')
).add_to(m)

In [5]:
for vehicle_id in routes['vehicle'].unique():
    vehicle_routes = routes[routes['vehicle'] == vehicle_id].sort_values('stop_sequence')
    color = colors[vehicle_id - 1]
    
    cumulative_distance = 0
    cumulative_time = 0
    prev_node = 0  # depot is always index 0
    
    for _, row in vehicle_routes.iterrows():
        current_node = int(row['node'])
        
        # distance from previous stop to this one
        leg_distance = distance_matrix[prev_node][current_node]
        cumulative_distance += leg_distance
        
        # time = driving time + 5 min service time per stop
        driving_time = (leg_distance / 1000) / 30 * 60
        cumulative_time += driving_time + 2
        
        popup_text = folium.Popup(f"""
    <b>Driver {vehicle_id} — Stop {int(row['stop_sequence'])}</b><br>
    <b>Address:</b> {row['address']}<br>
    <b>Arrival:</b> {round(cumulative_time, 1)} mins from depot<br>
    <b>Distance:</b> {round(cumulative_distance/1000, 2)} km<br>
    <b>Packages:</b> {int(sample.iloc[current_node]['demand'])}
""", max_width=250)
        
        folium.CircleMarker(
            location=[row['latitude'], row['longitude']],
            radius=8,
            color=color,
            fill=True,
            popup=popup_text
        ).add_to(m)
        
        prev_node = current_node
        
for vehicle_id in routes['vehicle'].unique():
    vehicle_routes = routes[routes['vehicle'] == vehicle_id].sort_values('stop_sequence')
    color = colors[vehicle_id - 1]

    nodes = vehicle_routes['node'].tolist()

    for i in range(len(nodes) - 1):
        orig_node = sample.iloc[nodes[i]]['node_id']
        dest_node = sample.iloc[nodes[i+1]]['node_id']

        try:
            route_nodes = nx.shortest_path(G, int(orig_node), int(dest_node), weight = 'length')
            route_coords = [[G.nodes[n]['y'], G.nodes[n]['x']] for n in route_nodes]

            folium.PolyLine(
                locations = route_coords,
                color = color,
                weight = 3, 
                opacity = 0.7
            ).add_to(m)
        except: 
            pass

In [6]:
m.save('../output/route_map.html')